# Tuần 6 — Chốt Train / Validation / Test theo bệnh nhân

Notebook tạo split ba tập không rò rỉ bệnh nhân và đóng gói hai biến thể dữ liệu: nhịp căn theo annotation chuẩn để đánh giá phân loại, và nhịp căn theo detector để dùng đánh giá end-to-end sau này.

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mitdb').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục mitdb ở thư mục hiện tại hoặc thư mục cha.')
WEEK3_DIR = PROJECT_DIR / 'Tuần 3-4' / 'ket_qua_ecg'
RESULT_DIR = PROJECT_DIR / 'Tuần 5-6-7' / 'ket_qua_tuan_6'
FIG_DIR = RESULT_DIR / 'figures'
DATA_DIR = PROJECT_DIR / 'mitdb' / 'mit-bih-arrhythmia-database-1.0.0' / 'mit-bih-arrhythmia-database-1.0.0'
RESULT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(exist_ok=True)

AAMI = {'N': set('N L R e j'.split()), 'S': set('A a J S'.split()),
        'V': set('V E'.split()), 'F': {'F'}, 'Q': set('/ f Q'.split())}
SYMBOL_TO_CLASS = {symbol: label for label, symbols in AAMI.items() for symbol in symbols}
CLASSES = ['N', 'S', 'V', 'F', 'Q']
PACED = {'102', '104', '107', '217'}
DS1 = set('101 106 108 109 112 114 115 116 118 119 122 124 201 203 205 207 208 209 215 220 223 230'.split())
DS2 = set('100 103 105 111 113 117 121 123 200 202 210 212 213 214 219 221 222 228 231 232 233 234'.split())
PATIENT = {record: record for record in DS1 | DS2}
PATIENT['201'] = PATIENT['202'] = 'patient_201_202'
TEST_RECORDS = DS2 - {'202'}
CANDIDATE_TRAIN_RECORDS = DS1 | {'202'}

def reference_beats(record):
    saved = np.load(WEEK3_DIR / record / 'cleaned.npz')
    signal, fs = saved['filtered'].astype(np.float32), float(saved['fs'])
    annotation = wfdb.rdann(str(DATA_DIR / record), 'atr')
    xs, ys, records = [], [], []
    for peak, symbol in zip(annotation.sample, annotation.symbol):
        label = SYMBOL_TO_CLASS.get(symbol)
        start, stop = int(peak) - 90, int(peak) + 90
        if label is not None and start >= 0 and stop <= len(signal):
            xs.append(signal[start:stop]); ys.append(label); records.append(record)
    return np.asarray(xs, dtype=np.float32).reshape(-1, 180), np.asarray(ys, dtype='U1'), np.asarray(records, dtype='U3')

# Đếm nhãn theo nhóm bệnh nhân để chọn validation có phân bố gần phần train nhất.
patient_records = {}
for patient in sorted({PATIENT[r] for r in CANDIDATE_TRAIN_RECORDS}):
    patient_records[patient] = sorted(r for r in CANDIDATE_TRAIN_RECORDS if PATIENT[r] == patient)
patient_counts = {}
for patient, records in patient_records.items():
    counts = Counter()
    for record in records:
        _, labels, _ = reference_beats(record)
        counts.update(labels.tolist())
    patient_counts[patient] = np.array([counts[c] for c in CLASSES], dtype=int)
all_counts = np.sum(list(patient_counts.values()), axis=0)
patients = sorted(patient_records)
# Chọn 4/22 nhóm bệnh nhân; score ưu tiên tỷ lệ lớp gần 18% dữ liệu train và có đủ lớp xuất hiện.
target_fraction = 4 / len(patients)
best = None
for chosen in combinations(patients, 4):
    val_counts = np.sum([patient_counts[p] for p in chosen], axis=0)
    if np.any((all_counts > 0) & (val_counts == 0)):
        continue
    class_ratio_error = np.abs(val_counts / np.maximum(all_counts, 1) - target_fraction).sum()
    size_error = abs(val_counts.sum() / all_counts.sum() - target_fraction)
    score = class_ratio_error + size_error
    if best is None or score < best[0]: best = (score, chosen, val_counts)
if best is None:
    raise RuntimeError('Không tìm được validation có đủ mọi lớp AAMI.')
_, VAL_PATIENTS, _ = best
VAL_RECORDS = {r for p in VAL_PATIENTS for r in patient_records[p]}
TRAIN_RECORDS = CANDIDATE_TRAIN_RECORDS - VAL_RECORDS

def patient_set(records): return {PATIENT[r] for r in records}
assert patient_set(TRAIN_RECORDS).isdisjoint(patient_set(VAL_RECORDS))
assert patient_set(TRAIN_RECORDS).isdisjoint(patient_set(TEST_RECORDS))
assert patient_set(VAL_RECORDS).isdisjoint(patient_set(TEST_RECORDS))
assert '201' in TRAIN_RECORDS or '201' in VAL_RECORDS
assert ('201' in VAL_RECORDS) == ('202' in VAL_RECORDS)

split_by_record = {r: 'train' for r in TRAIN_RECORDS} | {r: 'val' for r in VAL_RECORDS} | {r: 'test' for r in TEST_RECORDS}
split_rows = [{'record': r, 'patient_group': PATIENT[r], 'split': split_by_record[r],
               'note': '201/202 cùng bệnh nhân' if r in {'201', '202'} else ''}
              for r in sorted(split_by_record)]
splits = {'train': sorted(TRAIN_RECORDS), 'val': sorted(VAL_RECORDS), 'test': sorted(TEST_RECORDS)}
print('Danh sách validation được chọn:', sorted(VAL_RECORDS))
print('Nhóm bệnh nhân validation:', list(VAL_PATIENTS))
print({name: {'records': len(records), 'patients': len(patient_set(records))} for name, records in splits.items()})

def build_reference_dataset(records):
    parts = [reference_beats(record) for record in sorted(records)]
    return tuple(np.concatenate([part[i] for part in parts]) for i in range(3))

# Detector-aligned giữ nhịp hiện có của tuần 3–4; reference-aligned cắt quanh annotation chuẩn.
index_rows = list(csv.DictReader((WEEK3_DIR / 'beat_index.csv').open(encoding='utf-8-sig')))
symbol_by_peak = {(row['record'], int(row['r_peak_sample'])): row['symbol'] for row in index_rows}
def build_detector_dataset(records):
    xs, ys, rs = [], [], []
    for record in sorted(records):
        saved = np.load(WEEK3_DIR / record / 'beats_180.npz')
        for beat, peak in zip(saved['beats'], saved['r_peaks']):
            label = SYMBOL_TO_CLASS.get(symbol_by_peak.get((record, int(peak)), '?'))
            if label is not None:
                xs.append(beat); ys.append(label); rs.append(record)
    return np.asarray(xs, dtype=np.float32).reshape(-1, 180), np.asarray(ys, dtype='U1'), np.asarray(rs, dtype='U3')

reference, detector = {}, {}
distribution_rows = []
for split, records in splits.items():
    reference[split] = build_reference_dataset(records)
    detector[split] = build_detector_dataset(records)
    for source, dataset in [('reference_aligned', reference[split]), ('detector_aligned', detector[split])]:
        counts = Counter(dataset[1].tolist())
        for label in CLASSES:
            distribution_rows.append({'source': source, 'split': split, 'AAMI_class': label, 'beats': counts[label]})

distribution = pd.DataFrame(distribution_rows)
display_table = distribution.pivot(index=['source', 'split'], columns='AAMI_class', values='beats').fillna(0).astype(int)
print('Phân bố lớp:'); print(display_table.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for source, ax in zip(['reference_aligned', 'detector_aligned'], axes):
    table = distribution[distribution.source == source].pivot(index='AAMI_class', columns='split', values='beats').reindex(CLASSES).fillna(0)
    table[['train', 'val', 'test']].plot(kind='bar', ax=ax)
    ax.set_title(source.replace('_', ' ')); ax.set_ylabel('Số nhịp'); ax.set_xlabel('Lớp AAMI'); ax.tick_params(axis='x', rotation=0); ax.grid(axis='y', alpha=.25)
fig.suptitle('Phân bố lớp theo split — không trộn bệnh nhân', y=1.02); fig.tight_layout(); fig.savefig(FIG_DIR / 'phan_bo_lop_theo_split.png', dpi=180); plt.show()

split_counts = pd.DataFrame([{'split': split, 'records': len(records), 'patients': len(patient_set(records))} for split, records in splits.items()])
fig, ax = plt.subplots(figsize=(6, 4))
split_counts.set_index('split')[['records', 'patients']].plot(kind='bar', ax=ax)
ax.set_title('Quy mô split theo record và nhóm bệnh nhân'); ax.set_ylabel('Số lượng'); ax.tick_params(axis='x', rotation=0); ax.grid(axis='y', alpha=.25)
fig.tight_layout(); fig.savefig(FIG_DIR / 'record_va_benh_nhan_theo_split.png', dpi=180); plt.show()

pd.DataFrame(split_rows).to_csv(RESULT_DIR / 'split_train_val_test.csv', index=False, encoding='utf-8-sig')
distribution.to_csv(RESULT_DIR / 'class_distribution.csv', index=False, encoding='utf-8-sig')
for name, data in [('aami_reference_aligned', reference), ('aami_detector_aligned', detector)]:
    np.savez_compressed(RESULT_DIR / f'{name}.npz',
        x_train=data['train'][0], y_train=data['train'][1], record_train=data['train'][2],
        x_val=data['val'][0], y_val=data['val'][1], record_val=data['val'][2],
        x_test=data['test'][0], y_test=data['test'][1], record_test=data['test'][2],
        classes=np.asarray(CLASSES, dtype='U1'))
checks = {'train_records': splits['train'], 'val_records': splits['val'], 'test_records': splits['test'],
          'train_patients': sorted(patient_set(TRAIN_RECORDS)), 'val_patients': sorted(patient_set(VAL_RECORDS)),
          'test_patients': sorted(patient_set(TEST_RECORDS)), 'overlap_patients': [],
          'paced_records_excluded': sorted(PACED)}
(RESULT_DIR / 'split_validation_checks.json').write_text(json.dumps(checks, ensure_ascii=False, indent=2), encoding='utf-8')
(RESULT_DIR / 'dataset_policy.md').write_text('''# Chính sách Dataset v1 (tiền chuẩn hóa)\n\n- Chỉ dùng 44 record không paced; loại 102, 104, 107, 217.\n- Train/validation/test tách theo nhóm bệnh nhân; 201 và 202 luôn cùng split.\n- Nhãn supervised gồm N, S, V, F, Q; `?` và ký hiệu ngoài ánh xạ bị loại.\n- `reference_aligned` dùng annotation chuyên gia để đánh giá riêng mô hình phân loại.\n- `detector_aligned` dùng R-peak detector để đánh giá pipeline end-to-end sau này.\n''', encoding='utf-8')
print(f'Đã lưu kết quả tuần 6 vào: {RESULT_DIR}')
